# Optional extension: a toy signal in the frequency domain

**Optional extension after Sessions 1–3.** Whether you analyse gravitational-wave strain, pulsar timing or oscillating fields from a relativistic simulation, discrete Fourier transforms require strict attention to sampling units, observation duration and aliasing. Our noisy sinusoid is synthetic teaching data, not a physical waveform or detection claim.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u

plt.rcParams.update({'figure.figsize': (9.5, 3.4), 'font.size': 10.5})

## Sampling, frequency bins and Nyquist folding

Three rules govern a discrete real Fourier spectrum sampled at rate $f_s$ over total duration $T$ ($N = f_s T$ dimensionless samples):

1. **Bin spacing:** frequency resolution is set solely by observation duration, $\Delta f = 1 / T$ ($0.5\text{ Hz}$ for $T = 2\text{ s}$).
2. **One-sided real FFT (`np.fft.rfft`):** for real-valued time series, negative frequencies mirror positive ones, so `rfft` returns non-negative bins across $[0, f_s/2]$; scaling interior positive bins by $2/N$ recovers the sinusoid amplitude.
3. **Nyquist aliasing:** discrete samples at $f_s = 512\text{ Hz}$ resolve frequencies only up to $f_{\text{Nyq}} = f_s / 2 = 256\text{ Hz}$. Any real oscillation in $(f_{\text{Nyq}}, f_s)$ folds to $f_{\text{alias}} = f_s - f$ — so a $300\text{ Hz}$ wave sampled at $512\text{ Hz}$ appears at $212\text{ Hz}$.

![Diagram: Frequencies above Nyquist fold back into the resolved band](../book/diagrams/nyquist-aliasing.svg)

**Folding across $f_{\text{Nyq}}$.** Discrete samples at $512\text{ Hz}$ cannot distinguish $300\text{ Hz}$ from its $212\text{ Hz}$ alias without anti-alias filtering prior to digitisation. Below, the left panel illustrates that exact time-domain coincidence over $25\text{ ms}$, while the right panel recovers a $40\text{ Hz}$ injected tone in the `rfft` spectrum. [Editable source](https://github.com/xangma/jupyter_data_languages/blob/codex/phd-python-2026/book/diagrams/nyquist-aliasing.dot).

In [ ]:
rng = np.random.default_rng(14)
sample_rate = 512 * u.Hz
duration = 2 * u.s
injected_frequency = 40.0 * u.Hz

n_samples = int(round((sample_rate * duration).to_value(u.dimensionless_unscaled)))
sampling_hz = sample_rate.to_value(u.Hz)
duration_s = duration.to_value(u.s)
injected_hz = injected_frequency.to_value(u.Hz)
time = np.arange(n_samples) / sampling_hz

# Spectrum of a 40 Hz sinusoid in Gaussian noise
signal = np.sin(2 * np.pi * injected_hz * time) + rng.normal(0, 0.5, n_samples)
frequency = np.fft.rfftfreq(n_samples, d=1 / sampling_hz)
amplitude = 2 * np.abs(np.fft.rfft(signal)) / n_samples
nyquist = sampling_hz / 2
resolution = 1 / duration_s
peak = frequency[1:][np.argmax(amplitude[1:])]
predicted_alias = abs((injected_hz + nyquist) % sampling_hz - nyquist)
assert abs(peak - predicted_alias) <= resolution
print(f'Peak: {peak:.2f} Hz | predicted alias: {predicted_alias:.2f} Hz | bin spacing: {resolution:.2f} Hz')

# Time-domain aliasing illustration over the first 25 ms (300 Hz vs 212 Hz at 512 Hz sampling)
demo_high_hz = 300.0
demo_alias_hz = sampling_hz - demo_high_hz  # 212.0 Hz
t_fine = np.linspace(0, 0.025, 1000)
t_sub = time[time <= 0.025]
samples_sub = np.cos(2 * np.pi * demo_high_hz * t_sub)
np.testing.assert_allclose(samples_sub, np.cos(2 * np.pi * demo_alias_hz * t_sub), atol=1e-12)

fig, (ax_time, ax_freq) = plt.subplots(1, 2, figsize=(9.5, 3.4))
ax_time.plot(t_fine * 1e3, np.cos(2 * np.pi * demo_high_hz * t_fine), lw=1.0, alpha=0.75, label='300 Hz wave')
ax_time.plot(t_fine * 1e3, np.cos(2 * np.pi * demo_alias_hz * t_fine), '--', lw=1.6, label='212 Hz alias')
ax_time.plot(t_sub * 1e3, samples_sub, 'ko', ms=4.5, label='512 Hz samples')
ax_time.set(title='Time domain (first 25 ms)', xlabel='Time [ms]', ylabel='Amplitude')
ax_time.legend(fontsize=8, loc='lower left')

ax_freq.plot(frequency, amplitude, lw=1.2)
ax_freq.set(
    title=f'One-sided rfft spectrum ({injected_hz:g} Hz injected)',
    xlabel='Frequency [Hz]',
    ylabel='One-sided amplitude',
    xlim=(0, nyquist),
)
fig.tight_layout()
plt.show()

### Your turn

1. **Resolution:** Halve `duration` to `1 * u.s` — predict the new bin spacing $\Delta f$ before rerunning.
2. **Spectral leakage:** Set `injected_frequency = 40.3 * u.Hz` (between grid bins) and explain why amplitude spreads across neighbouring bins rather than peaking at `1.0`.
3. **Super-Nyquist injection:** Set `injected_frequency = 300.0 * u.Hz`, predict `predicted_alias` before rerunning, and check where the peak appears *(avoid exact multiples of $256\text{ Hz}$, where $\sin(2\pi f t_k) = 0$ at every sample).*

*(Note: DC and Nyquist bins do not take the interior factor-of-2 scaling, and real gravitational-wave searches require windowing, noise-PSD whitening and matched-filter significance rather than raw FFT peaks. Compare with [solutions](../book/solutions.md) when finished.)*

In [ ]:
# Your turn: experiment with duration = 1 * u.s, 40.3 * u.Hz, or 300.0 * u.Hz